In [4]:
import pandas as pd
from jobspy import scrape_jobs

search_terms = [
    "AI Engineer",
    "LLM Engineer",
    "Data Scientist",
    "Machine Learning Engineer"
]

all_scraped_jobs = []

for term in search_terms:
    print(f"Scraping positions for: {term}")

    try:
        jobs = scrape_jobs(
            site_name=["indeed"],
            search_term=term,
            #location="Bengaluru",
            country_indeed="India",
            results_wanted=100
        )

        if jobs.empty:
            print(f"No jobs found for {term}")
        else:
            print(f"Found {len(jobs)} jobs for {term}")

            # Add search term as a column
            jobs["searched_role"] = term

            all_scraped_jobs.append(jobs)

    except Exception as e:
        print(f"Error scraping {term}: {e}")

# Combine all results
if all_scraped_jobs:
    combined_jobs = pd.concat(all_scraped_jobs, ignore_index=True)

    columns_to_keep = [
        "searched_role",
        "site",
        "title",
        "company",
        "location",
        "date_posted",
        # "min_amount",
        # "max_amount",
        # "interval"
        "salary"
    ]

    filtered_jobs = combined_jobs[
        [col for col in columns_to_keep if col in combined_jobs.columns]
    ]

    # Remove duplicates
    filtered_jobs = filtered_jobs.drop_duplicates(
        subset=["title", "company", "location"]
    )

    filtered_jobs.to_csv("ai_jobs_bengaluru.csv", index=False)

    print(f"\nSaved {len(filtered_jobs)} jobs to ai_jobs_bengaluru.csv")
    print("Platforms scraped:", filtered_jobs["site"].unique())

else:
    print("No job data collected.")

Scraping positions for: AI Engineer
Found 100 jobs for AI Engineer
Scraping positions for: LLM Engineer
Found 100 jobs for LLM Engineer
Scraping positions for: Data Scientist
Found 100 jobs for Data Scientist
Scraping positions for: Machine Learning Engineer
Found 100 jobs for Machine Learning Engineer

Saved 272 jobs to ai_jobs_bengaluru.csv
Platforms scraped: ['indeed']


In [ ]:
import pandas as pd
import numpy as np
import re
from jobspy import scrape_jobs


search_terms = [
    "AI Engineer",
    "LLM Engineer",
    "Data Scientist",
    "Machine Learning Engineer"
]

all_scraped_jobs = []


for term in search_terms:

    print(f"\nScraping: {term}")

    try:
        jobs = scrape_jobs(
            site_name=["indeed"],
            search_term=f'"{term}"',
            country_indeed="India",
            results_wanted=100,
            description_format="markdown"
        )

        if jobs is None or jobs.empty:
            print(f"No jobs found for {term}")
            continue

        print(f"Found {len(jobs)} jobs")

        jobs["searched_role"] = term

        all_scraped_jobs.append(jobs)

    except Exception as e:
        print(f"Error scraping {term}: {e}")


if not all_scraped_jobs:
    print("No data collected.")
    exit()

combined_jobs = pd.concat(all_scraped_jobs, ignore_index=True)

print("\nTotal scraped jobs:", len(combined_jobs))


required_cols = [
    "min_amount",
    "max_amount",
    "interval",
    "description"
]

for col in required_cols:
    if col not in combined_jobs.columns:
        combined_jobs[col] = np.nan



def extract_salary_from_description(text):

    if pd.isna(text):
        return None

    text = str(text)

    patterns = [

        # ₹5,00,000 - ₹10,00,000
        r'₹\s*[\d,]+\s*-\s*₹\s*[\d,]+',

        # 5-10 LPA
        r'\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*LPA',

        # 5 LPA
        r'\d+(?:\.\d+)?\s*LPA',

        # CTC up to 12 LPA
        r'CTC.*?\d+(?:\.\d+)?\s*LPA',

        # ₹50,000 per month
        r'₹\s*[\d,]+\s*(?:per|a)\s*(?:month|year|annum)'
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(0)

    return None



def build_salary(row):

    min_amt = row["min_amount"]
    max_amt = row["max_amount"]
    interval = row["interval"]

    if pd.notna(min_amt) and pd.notna(max_amt):

        return f"₹{int(min_amt):,} - ₹{int(max_amt):,} {interval}"

    elif pd.notna(min_amt):

        return f"₹{int(min_amt):,}+ {interval}"

    elif pd.notna(max_amt):

        return f"Up to ₹{int(max_amt):,} {interval}"

    # fallback
    return extract_salary_from_description(
        row["description"]
    )

combined_jobs["salary"] = combined_jobs.apply(
    build_salary,
    axis=1
)


combined_jobs = combined_jobs.drop_duplicates(
    subset=[
        "title",
        "company",
        "location"
    ]
)



columns_to_keep = [
    "searched_role",
    "site",
    "title",
    "company",
    "location",
    "date_posted",
    "salary"
]

final_jobs = combined_jobs[
    [c for c in columns_to_keep
     if c in combined_jobs.columns]
]



salary_found = final_jobs["salary"].notna().sum()

print("\nJobs with salary info:", salary_found)
print("Jobs without salary info:", len(final_jobs) - salary_found)



final_jobs.to_csv(
    "ai_jobs_india.csv",
    index=False
)

print("\nSaved file: ai_jobs_india.csv")
print("Total jobs:", len(final_jobs))

print("\nSample rows:")
print(final_jobs.head())


Scraping: AI Engineer
Found 100 jobs

Scraping: LLM Engineer
Found 26 jobs

Scraping: Data Scientist
Found 100 jobs

Scraping: Machine Learning Engineer
Found 100 jobs

Total scraped jobs: 326

Jobs with salary info: 0
Jobs without salary info: 229

Saved file: ai_jobs_india.csv
Total jobs: 229

Sample rows:
  searched_role    site                        title    company location  \
0   AI Engineer  indeed  Senior AI Engineer - Remote  Jitterbit   TS, IN   
1   AI Engineer  indeed  Senior AI Engineer - Remote  Jitterbit   HR, IN   
2   AI Engineer  indeed  Senior AI Engineer - Remote  Jitterbit   KA, IN   
3   AI Engineer  indeed  Senior AI Engineer - Remote  Jitterbit   DL, IN   
4   AI Engineer  indeed  Senior AI Engineer - Remote       Zudy   DL, IN   

  date_posted salary  
0  2026-06-25   None  
1  2026-06-25   None  
2  2026-06-25   None  
3  2026-06-25   None  
4  2026-06-24   None  


In [ ]:
import pandas as pd
import numpy as np


def estimate_india_salary(row):
    
    if pd.notna(row['salary']) and str(row['salary']).strip() != "":
        return row['salary']
        
    title = str(row['title']).lower()
    role = str(row['searched_role']).lower()
    
    
    salary_bands = {
        "ai engineer": (8, 18, 35),
        "llm engineer": (12, 24, 42),
        "machine learning engineer": (9, 20, 38),
        "data scientist": (7, 16, 32)
    }
    
   
    base, senior, lead = salary_bands.get(role, (8, 16, 30))
    
    
    if any(word in title for word in ['intern', 'associate', 'analyst', '1', 'i ']):
        min_lpa, max_lpa = max(3, int(base * 0.6)), int(base * 1.0)
    elif any(word in title for word in ['principal', 'manager', 'director', 'gm', 'lead', 'architect', 'vp', 'avp']):
        min_lpa, max_lpa = int(lead * 0.8), int(lead * 1.4)
    elif any(word in title for word in ['senior', 'sr', 'ii', 'staff', 'expert']):
        min_lpa, max_lpa = int(senior * 0.85), int(senior * 1.3)
    else:
        
        min_lpa, max_lpa = int(base * 0.9), int(senior * 1.1)

    return f"₹{min_lpa}L - ₹{max_lpa}L PA "


final_jobs['salary'] = final_jobs.apply(estimate_india_salary, axis=1)


final_jobs.to_csv("ai_jobs_india_populated.csv", index=False)
print("Populated missing salaries using market tier estimates!")

Populated missing salaries using market tier estimates!


C:\Users\hp\AppData\Local\Temp\ipykernel_31512\830038651.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_jobs['salary'] = final_jobs.apply(estimate_india_salary, axis=1)
